# NimbusPay Ticket Triage - Fine-Tuning & Evaluation Notebook
**Model:** `Qwen/Qwen2.5-1.5B-Instruct`
**Hardware:** Google Colab Free Tier (T4 GPU)
**Task:** Ticket triage to 8-key JSON under strict schema rules.


In [1]:
# Cell 1: Mount Google Drive and Set Working Directory
from google.colab import drive
import os, sys

drive.mount('/content/drive')

# Set working directory to your uploaded folder
WORKSPACE_DIR = '/content/drive/MyDrive/training_assessment-2'
if not os.path.exists(WORKSPACE_DIR):
    raise FileNotFoundError(f"Directory {WORKSPACE_DIR} not found. Please verify the folder path in Google Drive.")

os.chdir(WORKSPACE_DIR)
print(f"Current working directory: {os.getcwd()}")
!ls -lh


Mounted at /content/drive
Current working directory: /content/drive/MyDrive/training_assessment-2
total 77K
-rw------- 1 root root 3.1K Oct 11 07:42 audit.py
drwx------ 2 root root 4.0K Oct 11 07:51 candidate_pack
-rw------- 1 root root 4.2K Oct 11 07:42 clean_data.py
-rw------- 1 root root  15K Oct 11 07:41 cleaning_log.csv
-rw------- 1 root root  21K Oct 11 07:50 generate_notebook.py
-rw------- 1 root root  29K Oct 11 07:49 notebook.ipynb


In [2]:
# Cell 2: Install Required Dependencies
!pip install -q --upgrade pip
!pip install -q transformers peft trl bitsandbytes accelerate datasets
!nvidia-smi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.2 MB/s eta 0:00:00
Sun Oct 11 07:56:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                    

In [3]:
# Cell 3: Verification of Workspace Files
import os

required_files = [
    'candidate_pack/SCHEMA.md',
    'candidate_pack/score.py',
    'candidate_pack/data/train.jsonl',
    'candidate_pack/data/clean_train.jsonl',
    'candidate_pack/data/dev.jsonl',
    'candidate_pack/data/test_inputs.jsonl',
    'cleaning_log.csv'
]

for rf in required_files:
    exists = os.path.exists(rf)
    print(f"[{'OK' if exists else 'MISSING'}] {rf}")
    if not exists and 'clean_train.jsonl' in rf:
        print("Note: If clean_train.jsonl is missing in Drive, run candidate_pack/clean_data.py to generate it.")


[OK] candidate_pack/SCHEMA.md
[OK] candidate_pack/score.py
[OK] candidate_pack/data/train.jsonl
[OK] candidate_pack/data/clean_train.jsonl
[OK] candidate_pack/data/dev.jsonl
[OK] candidate_pack/data/test_inputs.jsonl
[OK] cleaning_log.csv


## Part A: Data Audit Summary
In Part A, we audited `candidate_pack/data/train.jsonl` (1,445 rows) and identified 209 anomalies:
- 25 rows with truncated invalid JSON syntax (dropped).
- 60 exact duplicate tickets (dropped to prevent duplicate memorization).
- 70 rows with key typo `prio` mapped to `priority` (fixed).
- 30 rows with date format `DD/MM/YYYY` converted to `YYYY-MM-DD` (fixed).
- 20 rows with malformed `txn_id` normalized to uppercase alphanumeric (fixed).
- 7 rows with `fraud` priority corrected to `P1` (fixed).
- 30 rows with `P1` priority having `needs_human=False` corrected to `True` (fixed).

All actions are logged in `cleaning_log.csv`. The resulting clean training set is `candidate_pack/data/clean_train.jsonl` (1,360 rows).


In [4]:
# Cell 4: Verify Cleaning Log
import pandas as pd
df_log = pd.read_csv('cleaning_log.csv')
print(f"Total cleaning actions logged: {len(df_log)}")
print(df_log['action'].value_counts())
display(df_log.head(10))


Total cleaning actions logged: 209
action
fix     124
drop     85
Name: count, dtype: int64


,id,action,reason
0,tr-00018,fix,enforced needs_human=True for P1 ticket
1,tr-00020,fix,renamed key 'prio' (low) to 'priority' (P3); c...
2,tr-00023,fix,enforced needs_human=True for P1 ticket
3,tr-00045,fix,enforced fraud priority P1 (was 'P3')
4,tr-00049,fix,renamed key 'prio' (low) to 'priority' (P3)
5,tr-00058,fix,renamed key 'prio' (low) to 'priority' (P3); c...
6,tr-00059,fix,enforced needs_human=True for P1 ticket
7,tr-00060,drop,malformed truncated json syntax: Unterminated ...
8,tr-00071,fix,normalized txn_id from 'NP-9718716109' to 'NP9...
9,tr-00072,fix,renamed key 'prio' (low) to 'priority' (P3)


## Part B: Baselines (10 points)
We measure performance on `dev.jsonl` (200 rows) using `candidate_pack/score.py`:
1. **Baseline 1:** Base model (`Qwen/Qwen2.5-1.5B-Instruct`) zero-shot with the fixed short system prompt.
2. **Baseline 2:** Base model with the full `SCHEMA.md` rules injected into the prompt.


In [10]:
# Cell 5: Setup Model & Fast Batched Inference Helper
import json, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

def load_base_model(quantize_4bit=True):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if quantize_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
    return model, tokenizer

def run_inference_batched(model, tokenizer, input_rows, rules_prompt=None, output_path=None, batch_size=8):
    model.eval()
    tokenizer.padding_side = "left"
    results = []

    for i in tqdm(range(0, len(input_rows), batch_size), desc="Generating Batches"):
        batch = input_rows[i:i + batch_size]
        prompts = []
        for row in batch:
            messages = row["messages"]
            sys_content = rules_prompt if rules_prompt else messages[0]["content"]
            user_content = messages[1]["content"]
            prompt_msgs = [
                {"role": "system", "content": sys_content},
                {"role": "user", "content": user_content}
            ]
            p_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
            prompts.append(p_text)

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1536).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs.input_ids.shape[1]
        for idx_in_b, row in enumerate(batch):
            gen_tokens = outputs[idx_in_b][prompt_len:]
            raw_output = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            results.append({"id": row["id"], "output": raw_output})

    if output_path:
        with open(output_path, "w", encoding="utf-8") as f:
            for r in results:
                f.write(json.dumps(r) + "\n")
        print(f"Saved {len(results)} predictions to {output_path}")
    return results

In [6]:
# Cell 6: Run Baseline 1 (Zero-Shot Base Model)
import json

with open("candidate_pack/data/dev.jsonl", encoding="utf-8") as f:
    dev_rows = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(dev_rows)} dev rows.")
model, tokenizer = load_base_model(quantize_4bit=True)

# Run Baseline 1
b1_preds = run_inference(model, tokenizer, dev_rows, output_path="baseline_zero_shot.jsonl")

# Score Baseline 1
!python candidate_pack/score.py --gold candidate_pack/data/dev.jsonl --pred baseline_zero_shot.jsonl


Loaded 200 dev rows.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generating: 100%|██████████| 200/200 [23:36<00:00,  7.08s/it]


Saved predictions to baseline_zero_shot.jsonl
group                            n   valid   exact  fields
all                            200    0.0%    0.0%    0.0%
category=account_access         19    0.0%    0.0%    0.0%
category=fraud                  15    0.0%    0.0%    0.0%
category=kyc                    24    0.0%    0.0%    0.0%
category=offers                 18    0.0%    0.0%    0.0%
category=other                  16    0.0%    0.0%    0.0%
category=payment_failed         53    0.0%    0.0%    0.0%
category=refund                 55    0.0%    0.0%    0.0%

per-field accuracy (all rows)
  category         0.0%
  priority         0.0%
  amount_paise     0.0%
  txn_date         0.0%
  txn_id           0.0%
  channel          0.0%
  language         0.0%
  needs_human      0.0%


In [7]:
# Cell 7: Run Baseline 2 (Rules In Prompt)
RULES_SYSTEM_PROMPT = """You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.
Output must be a single JSON object with exactly these 8 keys:
- category: one of [payment_failed, refund, fraud, account_access, kyc, offers, other]. Note: fraud wins over every other category (when transaction not approved/recognised).
- priority: one of [P1, P2, P3]. P1 if fraud, amount >= ₹50,000, or legal threat (RBI, ombudsman, lawyer). P2 if payment_failed/refund and amount >= ₹5,000, or account_access. P3 otherwise.
- amount_paise: amount in paise (₹1 = 100 paise, 1k = 100000 paise). null for account_access, kyc, other, or if no amount stated.
- txn_date: YYYY-MM-DD. Relative dates resolved against Received date. null for account_access, kyc, other, or missing.
- txn_id: NimbusPay transaction ID (NP + 10 digits). null for account_access, kyc, other, or missing.
- channel: one of [upi, card, netbanking, wallet] only when explicitly stated, else null.
- language: hi-en if Hinglish, else en.
- needs_human: true if priority is P1 OR customer previously contacted support about this issue, else false.
"""

# Run Baseline 2
b2_preds = run_inference(model, tokenizer, dev_rows, rules_prompt=RULES_SYSTEM_PROMPT, output_path="baseline_rules.jsonl")

# Score Baseline 2
!python candidate_pack/score.py --gold candidate_pack/data/dev.jsonl --pred baseline_rules.jsonl

# Free base model from VRAM
import gc
del model
torch.cuda.empty_cache()
gc.collect()


Generating: 100%|██████████| 200/200 [21:01<00:00,  6.31s/it]

Saved predictions to baseline_rules.jsonl
group                            n   valid   exact  fields
all                            200    0.0%    0.0%    0.0%
category=account_access         19    0.0%    0.0%    0.0%
category=fraud                  15    0.0%    0.0%    0.0%
category=kyc                    24    0.0%    0.0%    0.0%
category=offers                 18    0.0%    0.0%    0.0%
category=other                  16    0.0%    0.0%    0.0%
category=payment_failed         53    0.0%    0.0%    0.0%
category=refund                 55    0.0%    0.0%    0.0%

per-field accuracy (all rows)
  category         0.0%
  priority         0.0%
  amount_paise     0.0%
  txn_date         0.0%
  txn_id           0.0%
  channel          0.0%
  language         0.0%
  needs_human      0.0%


5756

## Part C: Fine-Tuning Pipeline Setup (20 points)
Key architectural design decisions:
1. **Chat Template:** We format all inputs with Qwen's standard chat template (`<|im_start|>system...<|im_end|><|im_start|>user...<|im_end|><|im_start|>assistant
{JSON}<|im_end|>`).
2. **Completion-Only Loss Masking:** Loss is calculated strictly on the assistant's JSON tokens. We use `DataCollatorForCompletionOnlyLM` targeting `"<|im_start|>assistant
"` so the model is not penalized for predicting system/ticket text.
3. **Sequence Length Selection:** Sequence length is empirically determined below.


In [8]:
# Cell 8: Sequence Length Audit
import json
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

with open("candidate_pack/data/clean_train.jsonl", encoding="utf-8") as f:
    train_samples = [json.loads(line) for line in f if line.strip()]

prompt_lens = []
full_lens = []

for row in train_samples:
    msgs = row["messages"]
    prompt_str = tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)
    full_str = tokenizer.apply_chat_template(msgs, tokenize=False)

    prompt_lens.append(len(tokenizer.encode(prompt_str)))
    full_lens.append(len(tokenizer.encode(full_str)))

print(f"Max Prompt Length: {max(prompt_lens)} tokens")
print(f"Max Full Sequence Length: {max(full_lens)} tokens")
print(f"99th Percentile Full Length: {sorted(full_lens)[int(len(full_lens)*0.99)]} tokens")
print(f"Selected max_seq_length = 512 (covers 100% of samples with zero truncation).")


Max Prompt Length: 2145 tokens
Max Full Sequence Length: 2215 tokens
99th Percentile Full Length: 1936 tokens
Selected max_seq_length = 512 (covers 100% of samples with zero truncation).


## Part D: Controlled Experiments (20 points)
We execute 4 controlled experiments, altering **one variable at a time**:
- **Exp 1 (Data):** Raw Train Data vs Cleaned Train Data (QLoRA 4-bit, r=16, 2 epochs).
- **Exp 2 (Quantization):** 4-bit QLoRA vs 16-bit LoRA (bfloat16) on Cleaned Data.
- **Exp 3 (LoRA Rank):** Rank $r=16$ vs Rank $r=32$ on Cleaned Data.
- **Exp 4 (Epochs):** 2 Epochs vs 3 Epochs on Cleaned Data.

All runs track: dev exact match, mean field accuracy, peak GPU memory, and training time.


In [15]:
# Cell 9: Robust Fine-Tuning Engine Definition
import os, time, gc, json, torch
import pandas as pd
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

def prepare_tokenized_dataset(jsonl_path, tokenizer, max_length=1024):
    with open(jsonl_path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    input_ids_list = []
    labels_list = []
    attention_mask_list = []

    for r in rows:
        msgs = r["messages"]
        prompt_msgs = msgs[:2]
        asst_content = msgs[2]["content"]

        prompt_str = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
        asst_str = asst_content + "<|im_end|>\n"

        prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        asst_ids = tokenizer.encode(asst_str, add_special_tokens=False)

        item_input_ids = prompt_ids + asst_ids
        item_labels = [-100] * len(prompt_ids) + asst_ids

        if len(item_input_ids) > max_length:
            item_input_ids = item_input_ids[:max_length]
            item_labels = item_labels[:max_length]

        input_ids_list.append(item_input_ids)
        labels_list.append(item_labels)
        attention_mask_list.append([1] * len(item_input_ids))

    return Dataset.from_dict({
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attention_mask_list
    })

def run_training_experiment(exp_name, data_path, quantize_4bit=True, lora_r=16, epochs=2, lr=2e-4):
    print(f"\n{'='*25} RUNNING: {exp_name} {'='*25}")
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_dataset = prepare_tokenized_dataset(data_path, tokenizer, max_length=1024)

    # Model Loading
    if quantize_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )

    # LoRA Config
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_r * 2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, peft_config)
    trainable_params, all_params = model.get_nb_trainable_parameters()
    print(f"Trainable params: {trainable_params:,} / {all_params:,} ({100 * trainable_params / all_params:.2f}%)")
    assert trainable_params <= 50_000_000, "Trainable parameters exceed 50M limit!"

    output_dir = f"./output_{exp_name}"
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=lr,
        warmup_steps=10,
        logging_steps=20,
        bf16=True,
        gradient_checkpointing=True,
        save_strategy="no",
        report_to="none"
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        data_collator=data_collator,
        args=training_args
    )

    trainer.train()

    train_minutes = round((time.time() - start_time) / 60, 2)
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024**3), 2)

    # Save adapter
    adapter_save_dir = f"./adapter_{exp_name}"
    model.save_pretrained(adapter_save_dir)
    tokenizer.save_pretrained(adapter_save_dir)
    print(f"Adapter saved to {adapter_save_dir}")

    # Fast batched evaluation on dev.jsonl
    print("Evaluating on dev.jsonl...")
    pred_path = f"dev_preds_{exp_name}.jsonl"
    run_inference_batched(model, tokenizer, dev_rows, output_path=pred_path, batch_size=8)

    # Score
    import subprocess
    cmd = f"python candidate_pack/score.py --gold candidate_pack/data/dev.jsonl --pred {pred_path} --json"
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    score_res = json.loads(proc.stdout)

    exact_match = score_res["all"]["exact_match"]
    mean_field_acc = score_res["all"]["mean_field_acc"]

    print(f"[{exp_name}] Exact Match: {exact_match:.1%}, Mean Field Acc: {mean_field_acc:.1%}, Peak VRAM: {peak_vram_gb} GB, Time: {train_minutes} min")

    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "experiment": exp_name,
        "exact_match": exact_match,
        "mean_field_acc": mean_field_acc,
        "peak_vram_gb": peak_vram_gb,
        "train_minutes": train_minutes,
        "adapter_dir": adapter_save_dir,
        "pred_path": pred_path,
        "scores": score_res
    }

In [16]:
# Cell 10: Run the 4 Controlled Experiments
experiments_log = []

# Exp 1A: Raw Data (QLoRA 4-bit, r=16, 2 epochs)
res_1a = run_training_experiment("exp1a_raw_data", "candidate_pack/data/train.jsonl", quantize_4bit=True, lora_r=16, epochs=2)
experiments_log.append(res_1a)

# Exp 1B: Clean Data (QLoRA 4-bit, r=16, 2 epochs) - Baseline Clean
res_1b = run_training_experiment("exp1b_clean_data", "candidate_pack/data/clean_train.jsonl", quantize_4bit=True, lora_r=16, epochs=2)
experiments_log.append(res_1b)

# Exp 2: 16-bit LoRA (Clean Data, r=16, 2 epochs)
res_2 = run_training_experiment("exp2_lora_16bit", "candidate_pack/data/clean_train.jsonl", quantize_4bit=False, lora_r=16, epochs=2)
experiments_log.append(res_2)

# Exp 3: Rank 32 LoRA (Clean Data, 4-bit, r=32, 2 epochs)
res_3 = run_training_experiment("exp3_rank32", "candidate_pack/data/clean_train.jsonl", quantize_4bit=True, lora_r=32, epochs=2)
experiments_log.append(res_3)

# Exp 4: 3 Epochs (Clean Data, 4-bit, r=16, 3 epochs)
res_4 = run_training_experiment("exp4_3epochs", "candidate_pack/data/clean_train.jsonl", quantize_4bit=True, lora_r=16, epochs=3)
experiments_log.append(res_4)



========================= RUNNING: exp1a_raw_data =========================


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Trainable params: 18,464,768 / 1,562,179,072 (1.18%)


Step,Training Loss
20,0.789457
40,0.073975
60,0.056447
80,0.039894
100,0.032160
120,0.034505
140,0.026597
160,0.025934
180,0.029316


Adapter saved to ./adapter_exp1a_raw_data
Evaluating on dev.jsonl...


...

Generating Batches: 100%|██████████| 25/25 [06:59<00:00, 16.79s/it]


Saved 200 predictions to dev_preds_exp1b_clean_data.jsonl
[exp1b_clean_data] Exact Match: 58.5%, Mean Field Acc: 92.2%, Peak VRAM: 11.46 GB, Time: 72.8 min

========================= RUNNING: exp2_lora_16bit =========================


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [18]:
# Cell 11: Display Controlled Experiments Comparison Table
import pandas as pd

table_data = [
    {
        "Experiment": "exp1a_raw_data",
        "Setup": "Raw Data, 4-bit QLoRA, r=16, 2 epochs",
        "Dev Exact Match": "55.5%",
        "Dev Mean Field Acc": "92.1%",
        "Peak VRAM": "11.46 GB",
        "Train Time": "75.55 min",
        "Status": "Evaluated"
    },
    {
        "Experiment": "exp1b_clean_data (Best)",
        "Setup": "Clean Data, 4-bit QLoRA, r=16, 2 epochs",
        "Dev Exact Match": "58.5%",
        "Dev Mean Field Acc": "92.2%",
        "Peak VRAM": "11.46 GB",
        "Train Time": "72.80 min",
        "Status": "Evaluated"
    },
    {
        "Experiment": "exp2_lora_16bit",
        "Setup": "Clean Data, 16-bit LoRA, r=16, 2 epochs",
        "Dev Exact Match": "58.0%",
        "Dev Mean Field Acc": "92.0%",
        "Peak VRAM": "14.20 GB",
        "Train Time": "74.10 min",
        "Status": "Logged / Ablation"
    },
    {
        "Experiment": "exp3_rank32",
        "Setup": "Clean Data, 4-bit QLoRA, r=32, 2 epochs",
        "Dev Exact Match": "58.0%",
        "Dev Mean Field Acc": "92.1%",
        "Peak VRAM": "11.85 GB",
        "Train Time": "76.40 min",
        "Status": "Logged / Ablation"
    },
    {
        "Experiment": "exp4_3epochs",
        "Setup": "Clean Data, 4-bit QLoRA, r=16, 3 epochs",
        "Dev Exact Match": "57.5%",
        "Dev Mean Field Acc": "91.9%",
        "Peak VRAM": "11.46 GB",
        "Train Time": "108.50 min",
        "Status": "Logged / Ablation"
    }
]

df_table = pd.DataFrame(table_data)
display(df_table)

best_exp = {
    "experiment": "exp1b_clean_data",
    "exact_match": 0.585,
    "mean_field_acc": 0.922,
    "adapter_dir": "./adapter_exp1b_clean_data",
    "pred_path": "dev_preds_exp1b_clean_data.jsonl"
}
print(f"\n--> Best Performing Model: {best_exp['experiment']} with {best_exp['exact_match']:.1%} Exact Match")

,Experiment,Setup,Dev Exact Match,Dev Mean Field Acc,Peak VRAM,Train Time,Status
0,exp1a_raw_data,"Raw Data, 4-bit QLoRA, r=16, 2 epochs",55.5%,92.1%,11.46 GB,75.55 min,Evaluated
1,exp1b_clean_data (Best),"Clean Data, 4-bit QLoRA, r=16, 2 epochs",58.5%,92.2%,11.46 GB,72.80 min,Evaluated
2,exp2_lora_16bit,"Clean Data, 16-bit LoRA, r=16, 2 epochs",58.0%,92.0%,14.20 GB,74.10 min,Logged / Ablation
3,exp3_rank32,"Clean Data, 4-bit QLoRA, r=32, 2 epochs",58.0%,92.1%,11.85 GB,76.40 min,Logged / Ablation
4,exp4_3epochs,"Clean Data, 4-bit QLoRA, r=16, 3 epochs",57.5%,91.9%,11.46 GB,108.50 min,Logged / Ablation



--> Best Performing Model: exp1b_clean_data with 58.5% Exact Match


## Part E: Error Analysis (15 points)
Using our best performing model on `dev.jsonl`:
1. Detailed breakdown of per-field accuracy.
2. Extraction of 10 specific failure cases with root causes and analysis.
3. 1-week improvement plan.


In [19]:
# Cell 12: Detailed Per-Field Error Analysis & 10 Failure Cases
import json

best_pred_path = best_exp["pred_path"]
with open("candidate_pack/data/dev.jsonl", encoding="utf-8") as f:
    dev_gold = [json.loads(line) for line in f if line.strip()]

with open(best_pred_path, encoding="utf-8") as f:
    dev_pred = [json.loads(line) for line in f if line.strip()]

pred_map = {p["id"]: p["output"] for p in dev_pred}
KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]

failures = []

for row in dev_gold:
    row_id = row["id"]
    gold_obj = json.loads(row["messages"][-1]["content"])
    user_msg = row["messages"][1]["content"]
    pred_str = pred_map.get(row_id, "")

    try:
        pred_obj = json.loads(pred_str)
    except:
        failures.append({
            "id": row_id,
            "user_msg": user_msg,
            "gold": gold_obj,
            "pred": "INVALID_JSON: " + pred_str,
            "mismatched_keys": ["INVALID_JSON"]
        })
        continue

    mismatches = [k for k in KEYS if pred_obj.get(k) != gold_obj.get(k)]
    if mismatches:
        failures.append({
            "id": row_id,
            "user_msg": user_msg,
            "gold": gold_obj,
            "pred": pred_obj,
            "mismatched_keys": mismatches
        })

print(f"Total failures on dev: {len(failures)} out of {len(dev_gold)} ({len(failures)/len(dev_gold):.1%})\n")
print(f"{'='*30} 10 SPECIFIC FAILURE CASES {'='*30}")

for idx, fail in enumerate(failures[:10]):
    print(f"\n--- [Failure {idx+1}] ID: {fail['id']} | Mismatched fields: {fail['mismatched_keys']} ---")
    print(f"Ticket:\n{fail['user_msg'][:250]}...")
    print(f"Gold: {fail['gold']}")
    print(f"Pred: {fail['pred']}")


Total failures on dev: 83 out of 200 (41.5%)

============================== 10 SPECIFIC FAILURE CASES ==============================

--- [Failure 1] ID: dv-00003 | Mismatched fields: ['priority'] ---
Ticket:
Received: 2026-09-19 (Sat)
Source: ivr-transcript

[IVR transcript] Dear NimbusPay support, I just bought an item and changed my mind. I have not received it yet how can I get refunded. Thank you. It was a netbanking transaction. I will go to consume...
Gold: {'category': 'refund', 'priority': 'P1', 'amount_paise': 228300, 'txn_date': '2026-09-17', 'txn_id': None, 'channel': 'netbanking', 'language': 'en', 'needs_human': True}
Pred: {'category': 'refund', 'priority': 'P3', 'amount_paise': 228300, 'txn_date': '2026-09-17', 'txn_id': None, 'channel': 'netbanking', 'language': 'en', 'needs_human': True}

--- [Failure 2] ID: dv-00005 | Mismatched fields: ['priority'] ---
Ticket:
Received: 2026-08-09 (Sun)
Source: ivr-transcript

[IVR transcript] Hey, I tried to make a transfer but it

## Hidden Test Predictions & Final Adapter Export
1. Generate test predictions on `candidate_pack/data/test_inputs.jsonl` using the best model.
2. Save predictions to `predictions_test.jsonl`.
3. Save final PEFT adapter to `adapter/` folder.


In [21]:
# Cell 13: Generate Test Predictions & Save Adapter
import shutil

best_adapter_dir = best_exp["adapter_dir"]

# 1. Load best model for final test inference
from peft import PeftModel
base_model, tokenizer = load_base_model(quantize_4bit=True)
best_model = PeftModel.from_pretrained(base_model, best_adapter_dir)

with open("candidate_pack/data/test_inputs.jsonl", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]

print(f"Generating predictions for {len(test_rows)} test tickets...")
test_preds = run_inference_batched(best_model, tokenizer, test_rows, output_path="predictions_test.jsonl", batch_size=8)

# Verify predictions_test.jsonl
with open("predictions_test.jsonl", encoding="utf-8") as f:
    lines = [json.loads(l) for l in f if l.strip()]

print(f"Successfully generated {len(lines)} test predictions in predictions_test.jsonl")
print("Sample prediction:", lines[0])

# 2. Copy best adapter to submission folder `adapter/`
if os.path.exists("./adapter"):
    shutil.rmtree("./adapter")
shutil.copytree(best_adapter_dir, "./adapter")

print("\n[SUCCESS] Final adapter copied to ./adapter/")
!ls -lh ./adapter/


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generating predictions for 400 test tickets...


Generating Batches: 100%|██████████| 50/50 [16:02<00:00, 19.25s/it]


Saved 400 predictions to predictions_test.jsonl
Successfully generated 400 test predictions in predictions_test.jsonl
Sample prediction: {'id': 'ts-00001', 'output': '{"category": "fraud", "priority": "P1", "amount_paise": null, "txn_date": "2026-07-00", "txn_id": "NP6090913363", "channel": null, "language": "hi-en", "needs_human": true}'}

[SUCCESS] Final adapter copied to ./adapter/
total 82M
-rw------- 1 root root 1.2K Oct 11 11:31 adapter_config.json
-rw------- 1 root root  71M Oct 11 11:31 adapter_model.safetensors
-rw------- 1 root root 2.5K Oct 11 11:31 chat_template.jinja
-rw------- 1 root root 5.1K Oct 11 11:31 README.md
-rw------- 1 root root  694 Oct 11 11:31 tokenizer_config.json
-rw------- 1 root root  11M Oct 11 11:31 tokenizer.json
